# Referencia desagregada de hiperparámetros — entrenamiento global-local

Mismo orden de celdas que `train_model.ipynb`, pero con **todas** las funciones
expandidas: cada parámetro/flag explícito y comentado con `#` indicando qué controla.

Marcas:
- `# (NUEVO)`        parámetro añadido en la revisión de código.
- `# (NUEVO, OFF)`   implementado pero desactivado en el baseline (actívalo para calibrar).

No es necesario ejecutarlo tal cual: sirve como plantilla/diccionario de flags.


In [ ]:
# === 1. IMPORTS =============================================================
from pathlib import Path
import torch

from data.create_data import (
    build_local_dataloaders,          # loader de crops locales (LDLA)
    build_global_dataloaders,         # loader de cara completa (rama global)
    build_local_fused_dataloaders,    # loader alineado imagen->crops (fused loss)
    build_single_person_sampling_loader,  # loader fijo de monitoreo (1 persona)
)

import data.global_path_datasets as global_paths

from src.diffusion_pipeline.load_diffusion_models import (
    build_global_local_bundles,           # carga backbones SD (UNet/VAE/CLIP)
    build_mixed_lora_dora_training_setup, # inyecta LoRA/DoRA + optimizador
)
from src.score_net.load_scorenet import load_score_net_safely  # ScoreNet congelado
from src.loss.local_loss import LDLALocalAgingLoss             # pérdida local
from src.loss.global_loss import GlobalAgingLoss               # pérdida global
from src.loss.global_aux_bundle import GlobalLossAuxBundle     # ViT-edad + FaceNet (+LPIPS)
from src.training.train_aging_model import train_global_local_face_aging  # wrapper final
from src.training.mixed_precision import resolve_device, get_effective_amp_dtype


In [ ]:
# === 2. RUNTIME ============================================================
device      = resolve_device("auto")     # "cuda" si hay GPU, si no "cpu"
amp_enabled = True                        # activa autocast (mixed precision)
amp_dtype   = "bf16"                      # "bf16" (sin GradScaler) | "fp16" (con scaler)
dtype       = get_effective_amp_dtype(amp_dtype=amp_dtype, device=device) or torch.float32

run_name        = "notebook_global_local_run"                       # nombre del experimento
checkpoint_root = "training_checkpoints/notebook_global_local_run"  # carpeta de checkpoints


In [ ]:
# === 3. RUTAS DE DATOS GLOBALES (ajustar al host: Vast/Colab/local) =========
PROJECT_ROOT = Path("/workspace/dual-aging-diffusion")

global_paths.GLOBAL_IMAGE_DIR = PROJECT_ROOT / "data" / "global_extracted"   # dónde quedan las imágenes extraídas
global_paths.GLOBAL_CSV_PATH  = PROJECT_ROOT / "data" / "ffhq_predictions" / "ffhq_face_attribute_prompts.csv"  # CSV de edad/atributos
global_paths.DRIVE_ZIPS = [                                                   # ZIPs fuente del subset FFHQ ~6k
    PROJECT_ROOT / "data" / "global_zips" / "ffhq_subset_6k_extremes-001.zip",
    PROJECT_ROOT / "data" / "global_zips" / "ffhq_subset_6k_extremes-002.zip",
    PROJECT_ROOT / "data" / "global_zips" / "ffhq_subset_6k_extremes-003.zip",
    PROJECT_ROOT / "data" / "global_zips" / "ffhq_subset_6k_extremes-004.zip",
]
print("GLOBAL_CSV_PATH exists:", global_paths.GLOBAL_CSV_PATH.exists())


In [ ]:
# === 4. DATALOADERS ========================================================
skip_zones = ["labio_superior"]   # zonas locales excluidas en TODO el pipeline (crops/fused/sampling)

# --- 4.1 Loader local (crops) -----------------------------------------------
local_objects = build_local_dataloaders(
    batch_size = 4,            # tamaño de batch de crops
    num_workers = 4,           # procesos de carga
    pin_memory = True,         # pinned memory (acelera H2D)
    skip = skip_zones,         # zonas a excluir (alias de skip_regions)
    # skip_regions = None,     # equivalente a skip (nombres canónicos)
    # drop_regions = None,     # idem (compatibilidad)
)
# NOTA: estos hiperparámetros viven HARDCODE dentro de build_local_dataloaders
#       (edítalos ahí si los quieres cambiar):
#   val_fraction=0.15, seed=42                       -> split por image_id
#   LocalAgingCropDataset(train):
#       resolution=256, context_scale=1.20,
#       virtual_repeats=10                           -> repeticiones artificiales por época
#       jitter_scores_enabled=True                   -> (ya no afecta el prompt, ver Error 2)
#       enable_horizontal_flip=True, horizontal_flip_p=0.20   # (NUEVO) flip suave de textura
#       prompt_uses_original_score=True              # (NUEVO) prompt usa el score REAL (Error 2)
#   make_local_score_sampler (WeightedRandomSampler):
#       w_00_10=0.70 w_10_25=0.85 w_25_40=1.00 w_40_60=0.90
#       w_60_75=1.10 w_75_90=3.50 w_90_100=3.00      -> sobremuestreo de scores altos
#       region_balance_strength=0.35, use_anatomical_prior=True

# --- 4.2 Loader global (cara completa) --------------------------------------
global_objects = build_global_dataloaders(
    batch_size = 4,            # batch de caras completas (512x512)
    num_workers = 4,
    pin_memory = True,
    skip = skip_zones,         # (la rama global no tiene zonas; lo ignora)
)
# HARDCODE interno: min_age=None, max_age=None, age_col="age_pred",
#                   GLOBAL_RESOLUTION=512, train/val comparten 'samples' (revisión: split pendiente)

# --- 4.3 Loader fused (alineado imagen -> crops) ----------------------------
local_fused_objects = build_local_fused_dataloaders(
    batch_size = 1,                 # una cara completa con sus crops
    num_workers = 4,
    pin_memory = True,
    max_crops_per_image = None,     # None = todos los crops válidos por imagen
    skip = skip_zones,
)

# --- 4.4 Loader de monitoreo (1 persona fija) -------------------------------
sampling_objects = build_single_person_sampling_loader(
    image_stem = "09501",           # imagen fija para comparar épocas
    target_age = 75,                # edad objetivo global del sample
    skip = skip_zones,
    local_target_score = {          # score local objetivo por zona (default = no listadas)
        "default": 85.0,
        "frente": 85.0,
        "surcos_nasogenianos": 85.0,
        "bajo_ojo_ojeras": 85.0,
        "patas_de_gallo": 85.0,
    },
    num_workers = 2,
    pin_memory = False,
)

local_train_loader       = local_objects["train_loader"]
global_train_loader      = global_objects["train_loader"]
local_fused_train_loader = local_fused_objects["train_loader"]
monitor_loader           = sampling_objects["loader"]
sampling_loader_global   = monitor_loader
sampling_loader_local    = monitor_loader


In [ ]:
# === 5. BACKBONES (UNet/VAE/CLIP base) =====================================
global_bundle, local_bundle = build_global_local_bundles(
    global_model_id = "SG161222/Realistic_Vision_V6.0_B1_noVAE",  # checkpoint SD rama global
    global_vae_id   = "stabilityai/sd-vae-ft-mse",               # VAE externo global (modelo es noVAE)
    local_model_id  = "SG161222/Realistic_Vision_V6.0_B1_noVAE",  # checkpoint SD rama local
    local_vae_id    = None,        # None = reusa VAE del checkpoint local / global si es noVAE
    device          = device,      # colocación
    dtype           = dtype,       # precisión de carga
    print_memory    = True,        # reporta uso de VRAM al cargar
)


In [ ]:
# === 6. ADAPTERS + OPTIMIZADOR =============================================
mixed_global_bundle, mixed_local_bundle = build_mixed_lora_dora_training_setup(
    global_bundle = global_bundle,
    local_bundle  = local_bundle,

    global_adapter_config = {
        "adapter_type": "lora",    # tipo de adapter de la rama global
        "rank": 8,                 # rango LoRA (capacidad)
        "alpha": 8,                # escala LoRA
        "dropout": 0.0,            # dropout del adapter
        "target_suffixes": ["to_q","to_k","to_v","to_out.0","ff.net.0.proj","ff.net.2"],  # módulos UNet objetivo (atención+FFN)
    },
    local_adapter_config = {
        "adapter_type": "dora",    # DoRA en la rama local
        "rank": 16,                # más capacidad (texturas finas)
        "alpha": 16,
        "dropout": 0.05,           # leve regularización
        "target_suffixes": ["to_q","to_k","to_v","to_out.0","ff.net.0.proj","ff.net.2"],
    },
    optimizer_config = {
        "lr": 7e-5,                # learning rate (AdamW)
        "betas": (0.9, 0.999),     # momentos AdamW
        "weight_decay": 1e-2,      # weight decay
    },
    freeze_before_injection = True,  # congela VAE/UNet/CLIP; solo entrenan adapters
    print_memory = True,
    print_reports = True,
    verbose = True,
)


In [ ]:
# === 7. SCORENET (proxy de score local, congelado) =========================
score_net = load_score_net_safely(
    checkpoint_path = "models/score net/score_net_last_final.pt",  # pesos entrenados sobre crops reales
    device = str(device),
    dtype  = torch.float32,        # ScoreNet siempre en fp32 (evita NaN en gradiente auxiliar)
    base_channels = 32,            # debe coincidir con la arquitectura entrenada
    dropout = 0.15,                # idem
    strict = True,                 # exige match exacto de claves
    freeze = True,                 # congelado: solo evaluador en L_score / direccional
)


In [ ]:
# === 8. PÉRDIDA LOCAL (LDLA) ===============================================
local_loss = LDLALocalAgingLoss(
    local_bundle = mixed_local_bundle,   # bundle con UNet+DoRA local
    score_net    = score_net,            # evaluador de score (congelado)

    # --- Pesos de componentes ---
    lambda_full  = 1.0,    # reconstrucción condicionada (prompt fuente con score real)
    lambda_zone  = 0.15,   # reconstrucción condicionada (prompt solo de zona)
    lambda_score = 0.1,    # regresión MSE(ScoreNet(crop editado), score_target)
    lambda_cycle = 0.01,   # consistencia de ciclo (caro; pequeño)

    # --- Ventana de timesteps del camino de score ---
    score_timestep_min = 20,
    score_timestep_max = 350,

    freeze_score_net = True,
    device = str(device),

    # --- (NUEVO) Error 1: modo de generación del crop editado ---
    score_loss_mode        = "1_step_per_loss",  # "1_step_per_loss" (x0_hat 1 paso, borroso) | "full_ddim" (nítido)
    full_ddim_num_steps    = 10,                 # nº de pasos DDIM (solo si full_ddim)
    full_ddim_max_timestep = 120,                # t inicial bajo -> edición suave y nítida

    # --- (NUEVO, OFF) Error 10: Min-SNR en L_full / L_zone ---
    use_min_snr   = False,   # reponderación Min-SNR-gamma de la difusión
    min_snr_gamma = 5.0,     # gamma de Min-SNR

    # --- (NUEVO, OFF) Error 3: pérdida direccional / contrastiva de score ---
    use_directional_score  = False,  # activa la pérdida contrastiva (controlabilidad)
    lambda_direction       = 0.0,    # peso (>0 requerido para activar)
    direction_margin       = 0.05,   # gap mínimo exigido s_hi - s_lo (score normalizado)
    direction_high_score   = 0.90,   # token de score "alto" del par
    direction_low_score    = 0.30,   # token de score "bajo" del par
    direction_timestep_min = 20,     # ventana de ruido del par direccional
    direction_timestep_max = 200,
)


In [ ]:
# === 9. AUXILIARES GLOBALES (congelados) ===================================
global_aux_bundle = GlobalLossAuxBundle(
    device = str(device),
    dtype  = torch.float32,                 # auxiliares en fp32
    use_age = True,                         # estimador de edad (L_age / L_delta_age)
    age_model_id = "nateraw/vit-age-classifier",  # modelo ViT de edad por bins
    age_image_size = 224,                   # resolución de entrada del ViT
    use_identity = True,                    # encoder de identidad (L_id)
    identity_pretrained = "vggface2",       # pesos FaceNet
    identity_image_size = 160,              # resolución FaceNet
    use_lpips = False,                      # LPIPS perceptual (si True, lambda_perc puede ser >0)
)


In [ ]:
# === 10. PÉRDIDA GLOBAL ====================================================
global_loss = GlobalAgingLoss(
    global_bundle      = mixed_global_bundle,   # bundle UNet+LoRA global
    global_loss_bundle = global_aux_bundle,     # ViT-edad + FaceNet (+LPIPS)

    # --- Pesos ---
    lambda_diff      = 1.0,    # reconstrucción de difusión condicionada por edad
    lambda_id        = 0.35,   # preservación de identidad (FaceNet)
    lambda_age       = 0.15,   # edad absoluta -> edad objetivo
    lambda_delta_age = 0.15,   # cambio de edad coherente (Error 6: anclado a edad ViT de la fuente)
    lambda_perc      = 0.0,    # LPIPS (requiere use_lpips=True)

    age_loss_scale = 100.0,    # normaliza el error de edad (10 años -> 0.10)
    gamma_timestep = 1.0,      # exponente del peso por timestep en el camino semántico (modo 1_step)

    # --- Ventana de timesteps del camino semántico ---
    semantic_timestep_min = 20,
    semantic_timestep_max = 300,

    default_semantic_components = ("age", "delta_age", "id"),  # semánticas activas por defecto
    device = str(device),

    # --- (NUEVO) Error 1: modo de generación semántica ---
    semantic_loss_mode        = "1_step_per_loss",  # "1_step_per_loss" | "full_ddim"
    semantic_anchor_to_source = True,               # corrección RELATIVA (recon fuente vs recon target) en 1_step
    full_ddim_num_steps       = 10,                 # pasos DDIM (solo si full_ddim)
    full_ddim_max_timestep    = 120,                # t inicial bajo -> edición suave

    # --- (NUEVO, OFF) Error 10: Min-SNR en L_diff ---
    use_min_snr   = True,
    min_snr_gamma = 5.0,
)


In [ ]:
# === 11. ENTRENAMIENTO (wrapper de alto nivel) =============================
result = train_global_local_face_aging(
    # --- Objetos núcleo ---
    mixed_local_bundle  = mixed_local_bundle,
    mixed_global_bundle = mixed_global_bundle,
    local_train_loader  = local_train_loader,
    global_train_loader = global_train_loader,
    local_loss_fn  = local_loss,    # instancia (o usa *_loss_factory + *_loss_kwargs)
    global_loss_fn = global_loss,

    # --- Aux a mover CPU/GPU con cada rama (offload correcto) ---
    local_aux_objects  = [score_net],
    global_aux_objects = [global_aux_bundle],

    # --- Runtime ---
    device = device, amp_enabled = True, amp_dtype = "bf16",

    # --- Run / checkpoints ---
    run_name        = "notebook_global_local_run",
    checkpoint_root = "training_checkpoints/notebook_global_local_run",

    # --- Calendario ---
    num_epochs        = 6,                    # épocas por defecto
    local_num_epochs  = 6,                    # épocas rama local (puede diferir)
    global_num_epochs = 6,                    # épocas rama global
    start_epoch       = 0,
    train_order       = ("local", "global"),  # orden de ramas por época
    train_local       = True,
    train_global      = True,

    # --- Optimización ---
    local_grad_accum_steps  = 4,    # acumulación de gradiente local
    global_grad_accum_steps = 4,    # acumulación de gradiente global
    local_grad_clip  = 1.0,         # clip de norma local
    global_grad_clip = 1.0,         # clip de norma global

    # --- Muestreo de modos LOCAL ---
    local_p_full   = 0.45,   # prob. modo full (reconstrucción prompt fuente)
    local_p_score  = 0.40,   # prob. modo score (regresión de score)
    local_p_zone   = 0.15,   # prob. modo zone (prompt solo zona)
    local_enable_full  = True,
    local_enable_score = True,
    local_enable_zone  = True,
    local_p_neutral     = 0.05,  # dropout estilo CFG: usa prompt neutral (quita score)
    local_p_double_full = 0.20,  # prob. de doble-prompt explícito (source+neutral) en full

    local_p_direction      = 0.0,    # (NUEVO, OFF) prob. del modo direccional/contrastivo
    local_enable_direction = False,  # (NUEVO, OFF) requiere use_directional_score+lambda_direction en la loss
    local_p_uncond         = 0.0,    # (NUEVO, OFF) CFG real: prob. de prompt vacío ""

    # --- Fused loss local (opcional) ---
    local_fused_train_loader = local_fused_train_loader,
    use_fused_loss   = False,        # actívalo cuando quieras la fusión diferenciable
    fused_loss_epoch = 15,           # época a partir de la cual corre
    fused_loss_every_n_steps = 1,    # cada cuántos steps
    lambda_fuse_score = 0.03,        # peso score sobre cara fusionada
    lambda_fuse_seam  = 0.01,        # peso de costura
    fused_global_forward_fn = None,  # global congelado para x_global (None -> x_global=x_orig)

    # --- Muestreo de modos GLOBAL ---
    global_p_diff     = 0.65,        # prob. modo diff (reconstrucción)
    global_p_semantic = 0.35,        # prob. modo semántico (age/id/...)
    global_enable_diff     = True,
    global_enable_semantic = True,
    global_semantic_components = ("age", "delta_age", "id"),  # semánticas activas
    global_p_neutral    = 0.05,      # dropout estilo CFG (quita edad)
    global_p_double_diff = 0.15,     # doble-prompt en diff
    min_target_age = 18,             # edad objetivo mínima muestreada
    max_target_age = 90,             # edad objetivo máxima

    # --- Schedulers ---
    build_schedulers_if_missing = True,  # crea warmup+coseno si no hay
    local_warmup_ratio  = 0.05,
    global_warmup_ratio = 0.05,
    local_min_lr  = 1e-6,
    global_min_lr = 1e-6,
    min_warmup_steps = 10,
    max_warmup_steps = None,

    # --- Checkpoints ---
    save_latest = True,
    save_best   = True,
    save_inference_copy = True,
    local_monitor_key  = "loss/total",   # métrica para "best" (hoy loss de train)
    global_monitor_key = "loss/total",

    # --- Memoria ---
    enable_gradient_checkpointing_flag = True,  # checkpointing en UNet
    offload_after_each_branch = True,           # rama inactiva (y optimizer) a CPU
    print_memory = True,

    # --- Control de bucle (smoke tests) ---
    local_max_batches  = None,
    global_max_batches = None,

    # --- Sample de monitoreo (fijo) ---
    sampling_loader_global = sampling_loader_global,
    sampling_loader_local  = sampling_loader_local,
    sample_every_epochs    = 1,       # genera grid cada N épocas
    sample_after_epoch_zero = False,
    sampling_output_dir = "training_checkpoints/notebook_global_local_run/samples",

    # --- Sampling img2img: rama global ---
    sample_global_strength = 0.30,            # fuerza img2img global
    sample_global_guidance_scale = 4.5,       # CFG scale global
    sample_global_num_inference_steps = 45,   # pasos de inferencia global
    sample_global_negative_prompt = (
        "horror, zombie, corpse, skull, deformed face, distorted eyes, "
        "extreme wrinkles, diseased skin, low quality, artifacts"
    ),

    # --- Sampling img2img: crops locales ---
    sample_local_strength = 0.45,             # fuerza img2img local
    sample_local_guidance_scale = 2.0,        # CFG scale local
    sample_local_num_inference_steps = 45,
    sample_local_negative_prompt = (
        "blurry, smooth plastic skin, distorted skin, artifacts, low quality"
    ),
    sample_local_recycle_passes = 2,          # pasadas de reciclado local antes de fusionar
    sample_local_recycle_strength = 0.07,     # fuerza de cada pasada de reciclado
    # sample_local_recycle_guidance_scale = None,
    # sample_local_recycle_num_inference_steps = None,

    # --- Fusión determinista (sampling) ---
    sample_residual_alpha = 0.40,             # alpha del residuo global (cuánto se inyecta)
    sample_residual_sigma = 7.5,              # sigma del filtro gaussiano (baja frecuencia)
    sample_use_face_mask = True,              # restringe el residuo a la cara
    sample_face_mask_blur_sigma = 3.0,        # suavizado de la máscara facial
    sample_local_insert_alpha = 1.10,         # intensidad de inserción de crops
    sample_local_mask_blur_sigma = 4.0,       # feathering de máscaras locales
    sample_color_match = True,                # igualación de color del crop
    sample_color_match_strength = 0.60,
    sample_seed = 777,                        # semilla fija (comparar épocas)
    sample_save_grid = True,

    # --- Logging ---
    inner_print_every = 300,
    inner_verbose = False,
    print_first_batch = False,
    verbose = True,
)
result


## Variante: run solo-local

Para entrenar solo la rama local, cambia en la celda 11:

```python
global_loss_fn=None, train_global=False, global_num_epochs=0, global_max_batches=0,
train_order=("local",),
global_p_diff=0.0, global_p_semantic=0.0,
global_enable_diff=False, global_enable_semantic=False,
# y sube zone para corregir regiones:
local_p_full=0.40, local_p_score=0.35, local_p_zone=0.25, local_p_double_full=0.10,
```

## Activar las mejoras nuevas (calibración)

```python
# Error 1 (DDIM nítido):   semantic_loss_mode="full_ddim" / score_loss_mode="full_ddim"
# Error 3 (direccional):   loss  -> use_directional_score=True, lambda_direction=0.10
#                          train -> local_enable_direction=True, local_p_direction=0.15
# Error 10 (Min-SNR):      loss  -> use_min_snr=True
# Error 10 (CFG real):     train -> local_p_uncond=0.10
# Errores 2/6/8 ya van activos por defecto (sin flag).
```
